# 02 – Métricas do Lote 1 (L1)

OBS-01, OBS-02, OBS-03, QA-02, QA-03, QA-04 e CAL-01..CAL-04, segundo `spec/metodologia_metricas.md`.

- Lê `processed/<sensor>_<ano>.parquet` (gerado pelo notebook 01), **um ficheiro de cada vez e só com as colunas necessárias**.
- Guarda resultados parciais por ficheiro em `metrics/_partial_lote1/` e **retoma** se a sessão cair (use `FORCE = True` para refazer).
- No fim consolida em `metrics/l1_*.parquet` e escreve `metrics/_log_lote1.csv` e `metrics/_params_lote1.json`.
- O código das métricas vem do repo (`git clone` do GitHub público): `data/pipeline/metrics/`. Para atualizar, volte a correr a célula 1.

**Antes de correr:** o atalho `Sprinklers` tem de estar em *O Meu Drive* (ver `data/README.md`). O VIIRS é pesado: se faltar memória, ative *High-RAM* em *Runtime → Change runtime type*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Sprinklers')        # ajustar se o atalho tiver outro nome
PROCESSED = ROOT / 'processed'
METRICS = ROOT / 'metrics'
REPO_URL = 'https://github.com/Rui-Martins31/NASA_Space_Apps_2026_Sprinklers.git'
REPO = Path('/content/repo')


def ensure_repo():
    if (REPO / '.git').exists():
        subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', str(REPO), 'switch', 'metricas-pipeline'], check=True)
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'metricas-pipeline'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'metricas-pipeline', REPO_URL, str(REPO)], check=True)


ensure_repo()
sys.path.insert(0, str(REPO / 'data' / 'pipeline'))

assert PROCESSED.exists(), f"Nao encontro {PROCESSED}. Corra primeiro o notebook 01 e confirme o atalho 'Sprinklers'."
METRICS.mkdir(exist_ok=True)

In [ ]:
# Core calendar sources. S-NPP is processed separately in notebook 09.
SENSORS = {'modis', 'viirs-jpss1'}
YEARS = None        # None selects all available years for these sources.
FORCE = False
GRID_DEG = 0.1


In [ ]:
import json, re, time, datetime
import pandas as pd

from metrics import common
from metrics.counts import (
    count_invalid_coords, obs01_raw_counts, obs02_confidence_dist, obs03_filtered_counts,
    qa02_retention, qa03_cellday_grouped, qa04_redundancy,
)
from metrics.calendar_metrics import (
    cal_annual, cal_coverage, cal_daily, cal_monthly, eligible_cell_days, global_day_counts,
)

COLS = ['country', 'instrument', 'satellite', 'latitude', 'longitude', 'acq_date', 'type', 'confidence_class']
FILE_RE = re.compile(r'^(?P<sensor>modis|viirs-[a-z0-9]+)_(?P<year>\d{4})\.parquet$', re.I)
PART = METRICS / '_partial_lote1'
PART.mkdir(exist_ok=True)
ALLOWED_SENSORS = {'modis', 'viirs-jpss1'}
TABLE_NAMES = ['obs01', 'obs02', 'obs03', 'qa02', 'qa03', 'qa04', 'cell_days', 'day_counts']


def atomic_write(df, path):
    tmp = path.with_name(path.name + '.tmp')
    df.to_parquet(tmp, index=False)
    tmp.replace(path)


def process_file(sensor, year, path):
    """Calcula as tabelas por ficheiro e guarda-as em PART/<sensor>_<ano>/."""
    out = PART / f'{sensor}_{year}'
    out.mkdir(exist_ok=True)
    marker = out / '_done.json'
    marker.unlink(missing_ok=True)
    signature = source_signature(sensor, year, path)
    t0 = time.time()
    df = pd.read_parquet(path, columns=COLS)
    expected = 'MODIS' if sensor == 'modis' else 'VIIRS'
    if not df['instrument'].astype('string').str.upper().eq(expected).fillna(False).all():
        raise ValueError(f'Unexpected instrument in {path}')
    if sensor == 'viirs-jpss1':
        satellite = df['satellite'].astype('string').str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
        if not satellite.isin(['N20', 'NOAA20', 'J1', '1']).all():
            raise ValueError(f'Expected NOAA-20 satellite data in {path}')
    dates = common.parse_dates(df['acq_date'])
    info = dict(sensor=sensor, year=year, grid_deg=GRID_DEG, n_rows=len(df),
                n_invalid_dates=int(dates.isna().sum()),
                n_invalid_coords=count_invalid_coords(df),
                n_rows_outside_file_year=int((dates.dt.year.dropna() != year).sum()))
    tables = {
        'obs01': obs01_raw_counts(df), 'obs02': obs02_confidence_dist(df),
        'obs03': obs03_filtered_counts(df), 'qa02': qa02_retention(df),
        'qa03': qa03_cellday_grouped(df, GRID_DEG), 'qa04': qa04_redundancy(df, GRID_DEG),
        'cell_days': eligible_cell_days(df, GRID_DEG), 'day_counts': global_day_counts(df),
    }
    for name, tbl in tables.items():
        atomic_write(tbl, out / f'{name}.parquet')
    info['seconds'] = round(time.time() - t0, 1)
    info['signature'] = signature
    atomic_json(info, marker)
    return info


def atomic_json(value, path):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, sort_keys=True), encoding='utf-8')
    temporary.replace(path)


def source_signature(sensor, year, path):
    return dict(schema=2, sensor=sensor, year=year, source=str(path.resolve()),
                size=path.stat().st_size, mtime_ns=path.stat().st_mtime_ns,
                grid_deg=GRID_DEG, confidence=list(common.ELIGIBLE_CONFIDENCE),
                eligible_type=common.ELIGIBLE_TYPE)


def read_json(path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return None


def is_done(marker, sensor, year, path):
    info = read_json(marker)
    return (isinstance(info, dict) and info.get('signature') == source_signature(sensor, year, path)
            and all((marker.parent / f'{name}.parquet').is_file() for name in TABLE_NAMES))


In [ ]:
selected_sensors = ALLOWED_SENSORS if SENSORS is None else set(SENSORS)
unsupported = selected_sensors - ALLOWED_SENSORS
if unsupported:
    raise ValueError(f'Unsupported core sensors {sorted(unsupported)}. Use 09_har12_snpp.ipynb for S-NPP.')
files = {}
for p in sorted(PROCESSED.glob('*.parquet')):
    m = FILE_RE.match(p.name)
    if m:
        files[(m['sensor'].lower(), int(m['year']))] = p

todo = sorted(k for k in files if k[0] in selected_sensors and (YEARS is None or k[1] in YEARS))
if not todo:
    raise FileNotFoundError(f'No selected processed files in {PROCESSED}')
years = sorted({year for _, year in todo})
if YEARS is not None and set(YEARS) - set(years):
    raise FileNotFoundError(f'Missing selected years: {sorted(set(YEARS) - set(years))}')
print(f'{len(files)} processed files, {len(todo)} selected core files')

for sensor, year in todo:
    marker = PART / f'{sensor}_{year}' / '_done.json'
    if is_done(marker, sensor, year, files[(sensor, year)]) and not FORCE:
        print(f'[reuse] {sensor} {year}')
        continue
    info = process_file(sensor, year, files[(sensor, year)])
    print(info)

selected_dirs = [PART / f'{sensor}_{year}' for sensor, year in todo]
selected_info = [read_json(directory / '_done.json') for directory in selected_dirs]
if not any(info['n_rows'] > info['n_invalid_dates'] for info in selected_info):
    raise ValueError('Selected processed files contain no valid acquisition dates')
for year in years:
    out = PART / f'cal_daily_{year}.parquet'
    daily_marker = out.with_suffix('.json')
    year_keys = [(sensor, y) for sensor, y in todo if y == year]
    signatures = [source_signature(sensor, y, files[(sensor, y)]) for sensor, y in year_keys]
    signature = dict(schema=2, sources=signatures)
    if FORCE or not out.is_file() or read_json(daily_marker) != signature:
        daily_marker.unlink(missing_ok=True)
        compact = pd.concat([pd.read_parquet(PART / f'{sensor}_{y}' / 'cell_days.parquet')
                             for sensor, y in year_keys], ignore_index=True)
        atomic_write(cal_daily(compact), out)
        atomic_json(signature, daily_marker)
        print(f'[daily] {year}: {len(year_keys)} selected sensors')


In [ ]:
def collect(name):
    dirs = selected_dirs
    grids = {json.loads((d / '_done.json').read_text()).get('grid_deg') for d in dirs}
    assert grids <= {GRID_DEG}, f'parciais calculados com grelhas diferentes {grids}; corra com FORCE = True'
    return pd.concat([pd.read_parquet(d / f'{name}.parquet') for d in dirs], ignore_index=True)


KEYS = {
    'obs01': ['country', 'year', 'month', 'instrument', 'satellite'],
    'obs02': ['country', 'year', 'instrument', 'confidence_class'],
    'obs03': ['country', 'year', 'month', 'instrument', 'satellite'],
    'qa02': ['country', 'year', 'month', 'instrument', 'satellite'],
    'qa03': ['country', 'year', 'instrument', 'satellite'],
    'qa04': ['country', 'year', 'instrument'],
}
NAMES = {'obs01': 'l1_obs01_raw_counts', 'obs02': 'l1_obs02_confidence_dist',
         'obs03': 'l1_obs03_filtered_counts', 'qa02': 'l1_qa02_retention',
         'qa03': 'l1_qa03_cellday_grouped', 'qa04': 'l1_qa04_redundancy'}

final = {}
for key, name in NAMES.items():
    tbl = collect(key).sort_values(KEYS[key]).reset_index(drop=True)
    dup = tbl.duplicated(KEYS[key]).sum()
    assert dup == 0, f'{name}: {dup} chaves repetidas entre ficheiros (ver AVISO acima)'
    atomic_write(tbl, METRICS / f'{name}.parquet')
    final[name] = tbl
    print(f'{name}: {len(tbl)} linhas')

# calendario: cobertura (global), diaria, mensal, anual
cov = cal_coverage(collect('day_counts'))
cov = cov.loc[cov['date'].dt.year.isin(years)].reset_index(drop=True)
daily = pd.concat([pd.read_parquet(PART / f'cal_daily_{year}.parquet') for year in years], ignore_index=True)
daily = daily.sort_values(['country', 'date', 'series']).reset_index(drop=True)
assert not daily.duplicated(['country', 'date', 'series']).any(), 'cal_daily com chaves repetidas'
countries = sorted(final['l1_obs01_raw_counts']['country'].unique())
monthly = cal_monthly(daily, cov, countries)
monthly = monthly.loc[monthly['year'].isin(years)].reset_index(drop=True)
annual = cal_annual(monthly)
for name, tbl in {'l1_cal_coverage': cov, 'l1_cal_daily': daily,
                  'l1_cal_monthly': monthly, 'l1_cal_annual': annual}.items():
    atomic_write(tbl, METRICS / f'{name}.parquet')
    final[name] = tbl
    print(f'{name}: {len(tbl)} linhas')

# log e parametros
dirs = selected_dirs
log = pd.DataFrame([json.loads((d / '_done.json').read_text()) for d in dirs]).sort_values(['sensor', 'year'])
log.to_csv(METRICS / '_log_lote1.csv', index=False)
try:
    commit = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
except Exception:
    commit = ''
(METRICS / '_params_lote1.json').write_text(json.dumps({
    'sensors': sorted(selected_sensors), 'years': years,
    'selected_files': [files[key].name for key in todo],
    'grid_deg': GRID_DEG, 'eligible_type': common.ELIGIBLE_TYPE,
    'eligible_confidence': list(common.ELIGIBLE_CONFIDENCE),
    'repo_commit': commit, 'run_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')}, indent=1))
display(log)

## Verificação
Confirma as invariantes principais entre as tabelas (falha com `AssertionError` se alguma não bater).

In [ ]:
o1, o3 = final['l1_obs01_raw_counts'], final['l1_obs03_filtered_counts']
assert o1['n_raw'].sum() == (log['n_rows'] - log['n_invalid_dates']).sum(), 'OBS-01 != linhas com data valida'
assert list(o1[o1.columns[:-1]].itertuples(index=False)) == list(o3[o3.columns[:-1]].itertuples(index=False)), 'OBS-03 nao tem os grupos de OBS-01'
q3 = final['l1_qa03_cellday_grouped']
assert (q3['n_grouped'] == q3['n_eligible'] - q3['n_cell_days']).all() and (q3['n_grouped'] >= 0).all()
q4 = final['l1_qa04_redundancy']
assert (q4['n_redundant'] == q4['sum_cell_days_by_satellite'] - q4['n_cell_days_union']).all()
for s in ('modis', 'viirs', 'combined'):
    d = daily.loc[daily['series'] == s, 'n_cells'].sum()
    assert monthly.loc[monthly['series'] == s, 'cell_days'].sum() == d, f'mensal != diaria ({s})'
    assert annual.loc[annual['series'] == s, 'cell_days'].sum() == d, f'anual != mensal ({s})'
w = monthly.pivot_table(index=['country', 'year', 'month'], columns='series', values='cell_days').fillna(0)
if {'modis', 'viirs', 'combined'} <= set(w.columns):
    assert (w['combined'] <= w['modis'] + w['viirs']).all() and (w['combined'] >= w[['modis', 'viirs']].max(axis=1)).all()
print('OK: invariantes verificadas')
print('Linhas descartadas por data invalida:', int(log['n_invalid_dates'].sum()),
      '| por coordenadas invalidas (so celulas):', int(log['n_invalid_coords'].sum()))